# Motorbike Detection — YOLOv8

**Dataset:** Roboflow `motobike-detection-otper-h43mq` v1 (CC BY 4.0), export định dạng YOLOv8.

| Thuộc tính | Giá trị |
|---|---|
| Số lớp | 1 (`motobike`) |
| Kích thước ảnh | 640×640 (Roboflow đã auto-orient + resize *stretch*) |
| Train / Valid / Test | 1152 / 136 / 67 ảnh |
| Nhãn | YOLO txt: `class cx cy w h` (chuẩn hoá 0–1) |
| Augmentation có sẵn | Không |

**Những điểm cần lưu ý về dữ liệu**
- Tập **test không có nhãn**, nên chỉ dùng để chạy inference. Đánh giá định lượng dùng tập validation.
- Ảnh là các **frame cắt từ video** (`3J4A78xx_frame_NNNN`, `78xx_frame_NNNN`). Các frame liền nhau gần như giống hệt nhau.
- Một số ảnh bị **upload trùng nhiều lần** (khác hash `.rf.`, nhãn hơi khác nhau), có bản nằm cùng lúc ở train/valid/test, gây **rò rỉ dữ liệu**.

**Nội dung notebook**
1. Cài đặt & cấu hình
2. Tìm dataset
3. EDA nhãn
4. Kiểm tra chất lượng dữ liệu: thống kê ảnh, kiểm tra nhãn, phát hiện trùng lặp/rò rỉ
5. Preprocessing: làm sạch nhãn, loại trùng, chia lại train/val **theo video**, cấu hình và xem trước augmentation
6. Huấn luyện 2 thí nghiệm: **A** (split gốc của Roboflow) và **B** (dữ liệu đã làm sạch + split theo video)
7. So sánh kết quả, chọn ngưỡng confidence, phân tích lỗi
8. Inference trên test & lưu kết quả

> Các ô **Nhận xét** ghi số liệu từ lần chạy Kaggle của v1, 2026-10-08 (Tesla T4, ultralytics 8.4.174). Nếu chạy lại mà số thay đổi thì cập nhật các ô này.
> Notebook này đồng bộ 1–1 với `motorbike-detection-v2.ipynb` (bản tiếng Anh): cell thứ *k* ở hai bản có cùng nội dung.

## 1. Cài đặt & cấu hình

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
ultralytics.checks()
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os, re, glob, random, shutil, warnings
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import yaml
from PIL import Image
from IPython.display import display

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": .3})

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- Cấu hình huấn luyện ----
QUICK = os.environ.get("QUICK_TEST") == "1"     # chạy thử nhanh trên CPU, KHÔNG dùng trên Kaggle
MODEL    = "yolov8n.pt" if QUICK else "yolov8s.pt"   # yolov8n/s/m/l/x
EPOCHS   = 1 if QUICK else 100
IMGSZ    = 320 if QUICK else 640
BATCH    = 8 if QUICK else 16
PATIENCE = 25
FRACTION = 0.1 if QUICK else 1.0
WORKERS  = 0 if QUICK else 4
VAL_FRAC = 0.15          # tỉ lệ ảnh dành cho validation khi chia theo video

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
DS_ROOT = WORK / "datasets"
RUNS = WORK / "runs"
print("WORK:", WORK, "| QUICK:", QUICK)

## 2. Tìm dataset

In [ ]:
search_roots = [os.environ.get("DATA_DIR", ""), "/kaggle/input", str(Path.cwd()), str(Path.cwd() / "data")]
DATA_ROOT = None
for root in filter(None, search_roots):
    for y in sorted(glob.glob(f"{root}/**/data.yaml", recursive=True)):
        p = Path(y).parent
        if (p / "train" / "images").exists() and (p / "valid" / "images").exists():
            DATA_ROOT = p; break
    if DATA_ROOT: break
assert DATA_ROOT is not None, "Không tìm thấy dataset"
print("DATA_ROOT:", DATA_ROOT)

orig = yaml.safe_load(open(DATA_ROOT / "data.yaml"))
CLASS_NAMES = orig["names"]
print("Classes:", CLASS_NAMES)

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
SPLITS = ["train", "valid", "test"]

def list_images(split):
    return sorted(p for p in (DATA_ROOT / split / "images").glob("*") if p.suffix.lower() in IMG_EXT)

def label_path(img_path):
    return img_path.parent.parent / "labels" / (img_path.stem + ".txt")

def read_labels(img_path):
    # Trả về mảng (N,5) hoặc None nếu không có file nhãn
    lp = label_path(img_path)
    if not lp.exists():
        return None
    rows = [l.split() for l in lp.read_text().strip().splitlines() if l.strip()]
    return np.array(rows, dtype=float).reshape(-1, 5)

def frame_id(name):
    # '7825_frame_0001_png.rf.<hash>.jpg' -> '7825_frame_0001'
    return name.split(".rf.")[0].replace("_png", "").replace("_jpg", "")

def video_id(name):
    # gộp '3J4A7825' và '7825' về cùng một video '7825'
    m = re.search(r"(\d{4})_frame", name)
    return m.group(1) if m else name.split("_")[0]

meta = pd.DataFrame([
    dict(split=s, path=p, name=p.name, frame=frame_id(p.name), video=video_id(p.name))
    for s in SPLITS for p in list_images(s)
])
meta.groupby("split").size().to_frame("images")

## 3. EDA nhãn

In [ ]:
records, summary = [], []
for split in SPLITS:
    imgs = list_images(split)
    n_lab = n_empty = n_box = 0
    for ip in imgs:
        lab = read_labels(ip)
        if lab is None:
            continue
        n_lab += 1; n_empty += len(lab) == 0; n_box += len(lab)
        for c, cx, cy, w, h in lab:
            records.append(dict(split=split, img=ip.name, cls=int(c), cx=cx, cy=cy, w=w, h=h))
    summary.append(dict(split=split, images=len(imgs), label_files=n_lab, empty_labels=n_empty, boxes=n_box,
                        boxes_per_img=round(n_box / max(n_lab, 1), 2)))

boxes = pd.DataFrame(records)
boxes["area"] = boxes.w * boxes.h
boxes["aspect"] = boxes.h / boxes.w
display(pd.DataFrame(summary))
display(boxes.groupby("split")[["w", "h", "area", "aspect"]].describe().T.round(3))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 9))
per_img = boxes.groupby(["split", "img"]).size()
for s in ["train", "valid"]:
    ax[0, 0].hist(per_img[s], bins=np.arange(.5, per_img.max() + 1.5), alpha=.6, label=s, density=True)
ax[0, 0].set_title("Số box / ảnh"); ax[0, 0].legend()

for s, c in [("train", "C0"), ("valid", "C1")]:
    d = boxes[boxes.split == s]
    ax[0, 1].scatter(d.w * 640, d.h * 640, s=5, alpha=.4, c=c, label=s)
ax[0, 1].set_xlabel("w (px)"); ax[0, 1].set_ylabel("h (px)"); ax[0, 1].set_title("Kích thước box"); ax[0, 1].legend()

ax[0, 2].hist(boxes.aspect, bins=40, color="C2"); ax[0, 2].set_title("Tỉ lệ h/w của box")

ax[1, 0].hist(np.sqrt(boxes.area) * 640, bins=40, color="C3")
for t, lbl in [(32, "small<32"), (96, "medium<96")]:
    ax[1, 0].axvline(t, ls="--", c="k"); ax[1, 0].text(t, ax[1, 0].get_ylim()[1] * .9, lbl)
ax[1, 0].set_title("sqrt(diện tích) theo px (ngưỡng COCO)")

h = ax[1, 1].hist2d(boxes.cx, boxes.cy, bins=30, cmap="viridis"); ax[1, 1].invert_yaxis()
ax[1, 1].set_title("Heatmap vị trí tâm box"); plt.colorbar(h[3], ax=ax[1, 1])

vc = pd.crosstab(meta.video, meta.split)[SPLITS]
vc.plot.bar(stacked=True, ax=ax[1, 2]); ax[1, 2].set_title("Số ảnh theo video × split")
plt.tight_layout(); plt.show()

**Nhận xét (lần chạy Kaggle của v1, 2026-10-08)**
- Trung bình ~1,5 box/ảnh → cảnh thưa, phần lớn ảnh có 1–2 xe.
- Box trung vị: w ≈ 0,14, h ≈ 0,53 (≈ 90×340 px), tỉ lệ h/w ≈ 3,8 → box **cao và hẹp**.
- √diện tích ≈ 175 px → gần như toàn bộ là vật **lớn** theo ngưỡng COCO (> 96 px), không cần kỹ thuật riêng cho vật nhỏ.
- ⚠️ Roboflow resize về 640×640 kiểu **stretch** (không giữ tỉ lệ). Nếu ảnh gốc là 16:9 thì chiều ngang bị nén mạnh hơn chiều dọc, nên tỉ lệ 3,8 có thể một phần là do méo hình. Khi chạy trên frame video gốc phải resize **stretch** giống vậy trước khi đưa vào mô hình (Ultralytics mặc định letterbox, giữ tỉ lệ). Tập test đã được stretch sẵn nên phần 8 không bị ảnh hưởng.

In [ ]:
def draw_boxes(ax, img_path, lab, color="lime"):
    im = Image.open(img_path); W, H = im.size
    ax.imshow(im); ax.axis("off")
    if lab is not None:
        for c, cx, cy, w, h in lab:
            ax.add_patch(patches.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False, ec=color, lw=2))

def show_samples(split, n=8, cols=4):
    imgs = random.sample(list_images(split), min(n, len(list_images(split))))
    rows = int(np.ceil(len(imgs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
    for ax, ip in zip(np.ravel(axes), imgs):
        draw_boxes(ax, ip, read_labels(ip)); ax.set_title(ip.name[:28], fontsize=8)
    for ax in np.ravel(axes)[len(imgs):]: ax.axis("off")
    plt.suptitle(f"{split} — ground truth"); plt.tight_layout(); plt.show()

show_samples("train")

## 4. Kiểm tra chất lượng dữ liệu

### 4.1 Thống kê ảnh: độ sáng, độ tương phản, độ nét
- **Độ sáng** = trung bình mức xám; **độ tương phản** = độ lệch chuẩn mức xám; **độ nét** = phương sai Laplacian (thấp nghĩa là ảnh mờ).
- Nếu phân bố của các split lệch nhau hoặc trải rộng thì nên tăng augmentation màu (`hsv_v`, `hsv_s`).

In [ ]:
def img_stats(p):
    g = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    return g.mean(), g.std(), cv2.Laplacian(g, cv2.CV_64F).var()

meta[["brightness", "contrast", "sharpness"]] = np.array([img_stats(p) for p in meta.path])
display(meta.groupby("split")[["brightness", "contrast", "sharpness"]].describe().T.round(1))

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for i, col in enumerate(["brightness", "contrast", "sharpness"]):
    for s in SPLITS:
        v = meta.loc[meta.split == s, col]
        ax[i].hist(np.log10(v + 1) if col == "sharpness" else v, bins=40, alpha=.5, density=True, label=s)
    ax[i].set_title(col + (" (log10)" if col == "sharpness" else "")); ax[i].legend()
plt.tight_layout(); plt.show()

# Ảnh ở hai cực (tối nhất / sáng nhất / mờ nhất)
picks = [("Tối nhất", meta.nsmallest(2, "brightness")), ("Sáng nhất", meta.nlargest(2, "brightness")),
         ("Mờ nhất", meta.nsmallest(2, "sharpness"))]
fig, axes = plt.subplots(1, 6, figsize=(24, 4))
k = 0
for title, df in picks:
    for _, r in df.iterrows():
        draw_boxes(axes[k], r.path, read_labels(r.path)); axes[k].set_title(f"{title}\n{r[['brightness','sharpness']].astype(float).round(1).to_dict()}", fontsize=8); k += 1
plt.tight_layout(); plt.show()

**Nhận xét (lần chạy Kaggle của v1, 2026-10-08)**
- Độ sáng TB ≈ 100/255 (hơi tối), độ tương phản (std) ≈ 26 (thấp), độ nét (phương sai Laplacian) ≈ 640.
- Phân bố của train/valid/test gần như trùng nhau → **không có lệch phân bố** về ánh sáng hay độ nét giữa các split. Augmentation HSV vẫn có ích, nhưng mục đích là chuẩn bị cho camera/ánh sáng mới, không phải để bù lệch giữa các split.

### 4.2 Kiểm tra nhãn
Các lỗi được kiểm tra: tọa độ ngoài [0,1], box bị cắt ra ngoài ảnh, box quá nhỏ hoặc suy biến, box trùng nhau trong cùng ảnh, class id không hợp lệ, ảnh không có nhãn.

In [ ]:
def iou_xywh(a, b):
    ax1, ay1, ax2, ay2 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1, bx2, by2 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0, min(ax2, bx2) - max(ax1, bx1)), max(0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter + 1e-9)

MIN_WH = 0.005   # box nhỏ hơn 0.5% cạnh ảnh (~3px) xem như lỗi
issues = []
for _, r in meta[meta.split != "test"].iterrows():
    lab = read_labels(r.path)
    if lab is None or len(lab) == 0:
        issues.append(dict(split=r.split, name=r["name"], issue="empty_label")); continue
    for i, (c, cx, cy, w, h) in enumerate(lab):
        if int(c) not in range(len(CLASS_NAMES)): issues.append(dict(split=r.split, name=r["name"], issue="bad_class"))
        if not (0 <= cx <= 1 and 0 <= cy <= 1): issues.append(dict(split=r.split, name=r["name"], issue="center_out_of_range"))
        if cx - w / 2 < -1e-3 or cx + w / 2 > 1 + 1e-3 or cy - h / 2 < -1e-3 or cy + h / 2 > 1 + 1e-3:
            issues.append(dict(split=r.split, name=r["name"], issue="box_exceeds_image"))
        if w < MIN_WH or h < MIN_WH: issues.append(dict(split=r.split, name=r["name"], issue="degenerate_box"))
        for j in range(i + 1, len(lab)):
            if iou_xywh(lab[i, 1:], lab[j, 1:]) > 0.9: issues.append(dict(split=r.split, name=r["name"], issue="duplicate_box"))

issues = pd.DataFrame(issues, columns=["split", "name", "issue"])
print("Tổng số vấn đề:", len(issues))
display(pd.crosstab(issues.issue, issues.split) if len(issues) else "Không có vấn đề")

**Nhận xét (lần chạy Kaggle của v1, 2026-10-08)**: chỉ có 1 vấn đề — 1 ảnh train có file nhãn rỗng. YOLO coi đó là **ảnh nền (negative)**; giữ lại vì giúp giảm FP. Không có box tràn, suy biến, trùng hay class lỗi.

### 4.3 Phát hiện ảnh trùng lặp & rò rỉ giữa các split
- **Trùng tên frame** (cùng `video_frame_NNNN`, khác hash Roboflow) **không** có nghĩa là trùng ảnh, vì có nhiều video được đặt cùng số hiệu. Vì vậy cần kiểm tra **nội dung pixel**.
- **Trùng chính xác**: ảnh thumbnail 64×64 lệch trung bình < 3 mức xám, tức là cùng một ảnh được upload nhiều lần. Các bản này sẽ được loại ở phần 5.
- **Gần trùng (dHash 64-bit)**: khoảng cách Hamming ≤ 5, thường là các frame liền kề của cùng video.

In [ ]:
def dhash(p, size=8):
    g = Image.open(p).convert("L").resize((size + 1, size), Image.BILINEAR)
    a = np.asarray(g, dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).ravel())

H  = np.stack([dhash(p) for p in meta.path])                                         # (N, 8) uint8
TH = np.stack([cv2.resize(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE), (64, 64), interpolation=cv2.INTER_AREA)
               for p in meta.path]).astype(np.float32)                                # thumbnail

# Cặp ứng viên: cùng tên frame hoặc cùng dHash -> xác nhận bằng pixel; gom nhóm bằng union-find
parent = list(range(len(meta)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
cands = set()
for key in [meta.frame, pd.Series([h.tobytes() for h in H])]:
    for idx in pd.Series(range(len(meta))).groupby(key.values).groups.values():
        idx = list(idx); cands |= {(a, b) for k, a in enumerate(idx) for b in idx[k + 1:]}
for a, b in cands:
    if np.abs(TH[a] - TH[b]).mean() < 3:
        parent[find(a)] = find(b)
meta["dup_group"] = [find(i) for i in range(len(meta))]

same_name = meta.groupby("frame").filter(lambda g: len(g) > 1)
print(f"Nhóm trùng tên frame: {same_name.frame.nunique()} | trong đó thực sự trùng pixel: "
      f"{same_name.groupby('frame').dup_group.nunique().eq(1).sum()}")
g = meta.groupby("dup_group").agg(n=("name", "size"), splits=("split", lambda s: ",".join(sorted(set(s)))))
g = g[g.n > 1]
print(f"Nhóm ảnh trùng chính xác: {len(g)} nhóm, {g.n.sum()} ảnh -> loại được {g.n.sum() - len(g)} ảnh thừa")
display(g.splits.value_counts().rename("số nhóm").to_frame().rename_axis("split chứa bản sao"))

# Cùng một ảnh nhưng nhãn có khớp nhau không? (đo nhiễu gán nhãn)
noise = []
for _, grp in meta[meta.dup_group.isin(g.index) & (meta.split != "test")].groupby("dup_group"):
    labs = [read_labels(p) for p in grp.path]
    for a in range(len(labs)):
        for b in range(a + 1, len(labs)):
            A, B = labs[a], labs[b]
            ious = [max((iou_xywh(x[1:], y[1:]) for y in B), default=0) for x in A]
            noise.append(dict(same_count=len(A) == len(B), mean_iou=np.mean(ious) if ious else np.nan))
noise = pd.DataFrame(noise)
if len(noise):
    print(f"Cặp bản sao có cùng số box: {noise.same_count.mean():.1%} | IoU trung bình giữa 2 lần gán nhãn: {noise.mean_iou.mean():.3f}")
    noise.mean_iou.plot.hist(bins=20, figsize=(7, 3), title="IoU giữa các lần gán nhãn cùng một ảnh (nhiễu nhãn)"); plt.show()

In [ ]:
# --- Gần trùng (rò rỉ do frame liền kề) ---
def hamming(A, B):
    return np.unpackbits(A[:, None, :] ^ B[None, :, :], axis=-1).sum(-1)

tr = (meta.split == "train").values
D = {s: hamming(H[(meta.split == s).values], H[tr]) for s in ["valid", "test"]}
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for s, c in [("valid", "C1"), ("test", "C2")]:
    m = D[s].min(1)
    ax[0].hist(m, bins=np.arange(-.5, 30.5), alpha=.6, label=s, color=c)
    print(f"{s}: {np.mean(m <= 5) * 100:.1f}% ảnh có bản gần trùng (Hamming ≤ 5) trong train")
ax[0].axvline(5.5, ls="--", c="k"); ax[0].set_title("Khoảng cách Hamming nhỏ nhất tới tập train"); ax[0].legend()

# Ví dụ cặp valid–train gần trùng
vi = np.where(meta.split == "valid")[0]; ti = np.where(tr)[0]
order = np.argsort(D["valid"].min(1))[:3]
ax[1].axis("off")
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for k, i in enumerate(order):
    j = D["valid"][i].argmin()
    axes[0, k].imshow(Image.open(meta.path.iloc[vi[i]])); axes[0, k].set_title("valid: " + meta.name.iloc[vi[i]][:26], fontsize=8)
    axes[1, k].imshow(Image.open(meta.path.iloc[ti[j]])); axes[1, k].set_title(f"train (d={D['valid'][i, j]}): " + meta.name.iloc[ti[j]][:20], fontsize=8)
for a in axes.ravel(): a.axis("off")
plt.suptitle("Cặp valid–train gần trùng nhất"); plt.tight_layout(); plt.show()

**Kết luận phần 4** (lần chạy Kaggle của v1, 2026-10-08)

| Phát hiện | Số liệu | Ý nghĩa |
|---|---|---|
| Trùng tên frame | 182 nhóm, chỉ 43 nhóm trùng pixel | Không tin được tên file, phải so pixel |
| Ảnh trùng chính xác | 45 nhóm / 90 ảnh (45 ảnh thừa) | Cùng một ảnh upload 2 lần |
| …nằm vắt qua các split | 7 test–train, 4 train–valid, 1 test–valid | Rò rỉ trực tiếp |
| Nhiễu nhãn | 100% cùng số box, IoU TB 0,892 | Hai lần vẽ box cho cùng ảnh lệch nhau ~11% |
| Gần trùng với train (Hamming ≤ 5) | valid 24,3%, test 40,3% | Frame liền kề của cùng video |

- Chất lượng ảnh và nhãn nhìn chung ổn, nhưng split gốc **bị rò rỉ**: ảnh trùng y hệt và frame liền kề của cùng một video nằm ở cả train lẫn valid. Vì vậy mAP trên valid gốc sẽ **lạc quan quá mức**. Phần 5 xử lý vấn đề này.
- IoU giữa 2 lần gán nhãn chỉ ~0,89, nên ở ngưỡng IoU 0,9/0,95 ngay cả người gán nhãn cũng không tự khớp với chính mình → **mAP@0.5:0.95 có trần tự nhiên**; đừng kỳ vọng hay cố tối ưu quá mức chỉ số này.

## 5. Preprocessing

### 5.1 Làm sạch & tạo 2 phiên bản dataset
| Bước | A — split gốc | B — sạch + chia theo video |
|---|---|---|
| Clip box vào [0,1], bỏ box suy biến / trùng, bỏ class lỗi | ✓ | ✓ |
| Loại ảnh trùng nội dung (giữ bản có nhiều box nhất) | ✗ | ✓ |
| Chia train/val | giữ nguyên Roboflow | **GroupShuffleSplit theo video**, không video nào nằm ở cả 2 tập |

Thí nghiệm A làm baseline, cho thấy con số mAP "đẹp" nhưng bị rò rỉ. Thí nghiệm B cho ước lượng trung thực hơn về khả năng tổng quát hoá sang video mới.

In [ ]:
def clean_labels(lab):
    # Clip box vào ảnh, bỏ box suy biến, class lỗi và box trùng
    if lab is None or len(lab) == 0:
        return np.zeros((0, 5))
    out = []
    for c, cx, cy, w, h in lab:
        if int(c) not in range(len(CLASS_NAMES)):
            continue
        x1, y1 = np.clip(cx - w / 2, 0, 1), np.clip(cy - h / 2, 0, 1)
        x2, y2 = np.clip(cx + w / 2, 0, 1), np.clip(cy + h / 2, 0, 1)
        w, h = x2 - x1, y2 - y1
        if w < MIN_WH or h < MIN_WH:
            continue
        b = np.array([c, (x1 + x2) / 2, (y1 + y2) / 2, w, h])
        if any(iou_xywh(b[1:], o[1:]) > 0.9 for o in out):
            continue
        out.append(b)
    return np.array(out).reshape(-1, 5)

def build_dataset(name, assignment):
    # assignment: list of (img_path, subset) với subset in {'train','val'}
    root = DS_ROOT / name
    if root.exists(): shutil.rmtree(root)
    n_box = 0
    for p, sub in assignment:
        (root / sub / "images").mkdir(parents=True, exist_ok=True)
        (root / sub / "labels").mkdir(parents=True, exist_ok=True)
        shutil.copy(p, root / sub / "images" / p.name)
        lab = clean_labels(read_labels(p))
        n_box += len(lab)
        (root / sub / "labels" / (p.stem + ".txt")).write_text(
            "\n".join(f"{int(c)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in lab))
    cfg = dict(path=str(root), train="train/images", val="val/images", nc=len(CLASS_NAMES), names=CLASS_NAMES)
    yaml.safe_dump(cfg, open(root / "data.yaml", "w"), sort_keys=False)
    print(f"[{name}] train={sum(s == 'train' for _, s in assignment)} val={sum(s == 'val' for _, s in assignment)} boxes={n_box}")
    return root / "data.yaml"

labelled = meta[meta.split != "test"].copy()

# ---- A: split gốc ----
YAML_A = build_dataset("A_original", [(r.path, "train" if r.split == "train" else "val") for _, r in labelled.iterrows()])

# ---- B: loại trùng + chia theo video ----
from sklearn.model_selection import GroupShuffleSplit
labelled["n_box"] = [0 if (l := read_labels(p)) is None else len(l) for p in labelled.path]
dedup = (labelled.sort_values(["n_box", "split", "name"], ascending=[False, True, True])
         .drop_duplicates("dup_group", keep="first").reset_index(drop=True))
print(f"Loại trùng: {len(labelled)} -> {len(dedup)} ảnh")

# Thử nhiều seed, chọn split có tỉ lệ val gần VAL_FRAC nhất và val có >= 2 video
best = None
for seed in range(500):
    gss = GroupShuffleSplit(n_splits=1, test_size=VAL_FRAC, random_state=seed)
    tr_idx, va_idx = next(gss.split(dedup, groups=dedup.video))
    frac = len(va_idx) / len(dedup)
    if dedup.video.iloc[va_idx].nunique() >= 2 and (best is None or abs(frac - VAL_FRAC) < best[0]):
        best = (abs(frac - VAL_FRAC), tr_idx, va_idx)
_, tr_idx, va_idx = best
dedup["subset"] = "train"; dedup.loc[va_idx, "subset"] = "val"
VAL_VIDEOS = sorted(dedup.loc[va_idx, "video"].unique())
print("Video dùng làm val:", VAL_VIDEOS, f"| tỉ lệ val = {len(va_idx) / len(dedup):.1%}")
YAML_B = build_dataset("B_video_split", list(zip(dedup.path, dedup.subset)))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
A_sub = labelled.assign(subset=np.where(labelled.split == "train", "train", "val"))
pd.crosstab(A_sub.video, A_sub.subset).plot.bar(stacked=True, ax=ax[0], color=["C0", "C1"])
ax[0].set_title("A — split gốc: video nào cũng có ở cả train và val")
pd.crosstab(dedup.video, dedup.subset).reindex(columns=["train", "val"], fill_value=0).plot.bar(stacked=True, ax=ax[1], color=["C0", "C1"])
ax[1].set_title("B — chia theo video: không chồng lấn")

def box_counts(yaml_path):
    root = Path(yaml_path).parent
    return {s: sum(len(open(f).read().split("\n")) if open(f).read().strip() else 0 for f in (root / s / "labels").glob("*.txt")) for s in ["train", "val"]}
cmp = pd.DataFrame({"A_original": box_counts(YAML_A), "B_video_split": box_counts(YAML_B)}).T
cmp.plot.bar(ax=ax[2], rot=0); ax[2].set_title("Số box train/val sau preprocessing")
plt.tight_layout(); plt.show()
display(cmp)

**Kết luận phần 5** (lần chạy Kaggle của v1, 2026-10-08)

| | A — split gốc | B — sạch + chia theo video |
|---|---|---|
| Ảnh train / val | 1152 / 136 | 1062 / 189 |
| Box train / val | 1723 / 201 | 1501 / 369 |
| Box/ảnh train / val | 1,50 / 1,48 | 1,41 / **1,95** |
| Video dùng làm val | mọi video | `7817`, `7825` (15,1% số ảnh) |

- Loại trùng: 1288 → 1251 ảnh (bớt 37 chứ không phải 45, vì một số bản trùng nằm ở tập test, vốn không có nhãn).
- ⚠️ **Val B đông xe hơn train** (1,95 so với 1,41 box/ảnh) → val B có thể khó hơn, không chỉ đơn giản là hết rò rỉ. Chênh lệch A − B ở mục 7.2 = rò rỉ **+** khác biệt độ khó, không thuần là rò rỉ.
- ⚠️ **Val B chỉ có 2 video** → ước lượng có phương sai cao; chọn 2 video khác có thể cho mAP khác hẳn. Muốn chắc chắn hơn: cross-validation theo video (GroupKFold).
- ⚠️ Ảnh đã bị **stretch** (xem nhận xét phần 3): khi deploy trên video gốc, tiền xử lý phải giống lúc train.

### 5.2 Augmentation (online, do Ultralytics thực hiện khi train)
Lựa chọn dựa trên kết quả EDA:
- Dataset nhỏ (~1k ảnh) và ít bối cảnh (một vài camera cố định), nên dùng **mosaic + scale + translate** mạnh để tăng đa dạng bối cảnh và kích thước.
- Box cao và hẹp (xe máy nhìn chính diện/nghiêng): dùng **lật ngang**, **không lật dọc**, xoay nhẹ ±5°.
- Độ sáng giữa các split gần giống nhau (mục 4.1) nhưng trong mỗi split trải rộng (≈ 57–149), và video/camera mới có thể khác: dùng **HSV** (đặc biệt `hsv_v`) để mô hình bền với ánh sáng.
- `mixup` nhẹ để giảm overfit; `close_mosaic` tắt mosaic ở 10 epoch cuối cho mô hình quen với ảnh thật.

Chuẩn hoá pixel về [0,1] và letterbox về `imgsz` do Ultralytics tự làm.

In [ ]:
AUG = dict(
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.45,
    degrees=5.0, translate=0.1, scale=0.5, shear=2.0, perspective=0.0,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.1, close_mosaic=10,
)

def preview_augmentation(data_yaml, n=8, cols=4):
    from ultralytics.cfg import get_cfg
    from ultralytics.data import build_yolo_dataset
    from ultralytics.data.utils import check_det_dataset
    data = check_det_dataset(str(data_yaml))
    cfg = get_cfg(overrides={**AUG, "imgsz": IMGSZ, "data": str(data_yaml)})
    ds = build_yolo_dataset(cfg, data["train"], batch=BATCH, data=data, mode="train")
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 4.5 * rows))
    for ax, i in zip(np.ravel(axes), random.sample(range(len(ds)), n)):
        s = ds[i]
        img = s["img"].permute(1, 2, 0).numpy()[..., ::-1]     # CHW BGR -> HWC RGB
        Hh, Ww = img.shape[:2]
        ax.imshow(img); ax.axis("off")
        for cx, cy, w, h in s["bboxes"].numpy():
            ax.add_patch(patches.Rectangle(((cx - w / 2) * Ww, (cy - h / 2) * Hh), w * Ww, h * Hh, fill=False, ec="yellow", lw=1.5))
    plt.suptitle("Ảnh train sau augmentation (mosaic, HSV, flip, scale, mixup...)"); plt.tight_layout(); plt.show()

try:
    preview_augmentation(YAML_B)
except Exception as e:
    print("Không xem trước được augmentation:", repr(e), "— xem train_batch*.jpg sau khi train.")

## 6. Huấn luyện
Hai thí nghiệm dùng cùng mô hình, cùng siêu tham số và cùng augmentation, **chỉ khác dữ liệu**:
- **A**: split gốc (baseline, bị rò rỉ)
- **B**: dữ liệu sạch, chia theo video

In [ ]:
from ultralytics import YOLO

def train_run(name, data_yaml):
    model = YOLO(MODEL)
    model.train(
        data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
        optimizer="auto", cos_lr=True, seed=SEED, deterministic=True,
        project=str(RUNS), name=name, exist_ok=True, plots=True,
        workers=WORKERS, fraction=FRACTION, **AUG,
    )
    return Path(model.trainer.save_dir)
print("Đã định nghĩa xong hàm train_run!")

In [ ]:
RUN_DIRS = {}
RUN_DIRS["A_original"] = train_run("A_original", YAML_A)

In [ ]:
RUN_DIRS["B_video_split"] = train_run("B_video_split", YAML_B)
RUN_DIRS

## 7. Đánh giá
### 7.1 Đường cong huấn luyện: so sánh A và B, kiểm tra overfitting

In [ ]:
hist = {}
for k, d in RUN_DIRS.items():
    r = pd.read_csv(d / "results.csv"); r.columns = r.columns.str.strip(); hist[k] = r

fig, ax = plt.subplots(2, 3, figsize=(20, 9))
plots = [("train/box_loss", "val/box_loss", "Box loss"), ("train/cls_loss", "val/cls_loss", "Cls loss"),
         ("train/dfl_loss", "val/dfl_loss", "DFL loss")]
for i, (tr_k, va_k, t) in enumerate(plots):
    for k, r in hist.items():
        ax[0, i].plot(r.epoch, r[tr_k], "--", label=f"{k} train")
        ax[0, i].plot(r.epoch, r[va_k], "-", label=f"{k} val")
    ax[0, i].set_title(t + " (nét đứt = train, nét liền = val)"); ax[0, i].legend(fontsize=8)
for i, (m, t) in enumerate([("metrics/mAP50(B)", "mAP@0.5"), ("metrics/mAP50-95(B)", "mAP@0.5:0.95")]):
    for k, r in hist.items():
        ax[1, i].plot(r.epoch, r[m], label=k)
    ax[1, i].set_title(t); ax[1, i].legend()
for k, r in hist.items():
    ax[1, 2].plot(r.epoch, r["metrics/precision(B)"], label=f"{k} P")
    ax[1, 2].plot(r.epoch, r["metrics/recall(B)"], "--", label=f"{k} R")
ax[1, 2].set_title("Precision / Recall"); ax[1, 2].legend(fontsize=8)
plt.tight_layout(); plt.show()

gap = pd.DataFrame({k: {"best_epoch": int(r.loc[r["metrics/mAP50-95(B)"].idxmax(), "epoch"]),
                        "epochs_run": len(r),
                        "val/box_loss − train/box_loss (cuối)": round(r["val/box_loss"].iloc[-1] - r["train/box_loss"].iloc[-1], 4)}
                    for k, r in hist.items()}).T
display(gap)

### 7.2 Metric trên validation với `best.pt`

In [ ]:
YAMLS = {"A_original": YAML_A, "B_video_split": YAML_B}
METRICS, MODELS = {}, {}
for k, d in RUN_DIRS.items():
    MODELS[k] = YOLO(str(d / "weights" / "best.pt"))
    METRICS[k] = MODELS[k].val(data=str(YAMLS[k]), imgsz=IMGSZ, batch=BATCH, project=str(RUNS),
                               name=f"val_{k}", exist_ok=True, plots=True, verbose=False)

score = pd.DataFrame({k: {"precision": m.box.mp, "recall": m.box.mr, "mAP@0.5": m.box.map50,
                          "mAP@0.75": m.box.map75, "mAP@0.5:0.95": m.box.map}
                      for k, m in METRICS.items()}).T.round(4)
display(score)
ax = score.T.plot.bar(rot=0, figsize=(10, 4), title="A (split gốc, rò rỉ) vs B (chia theo video)")
for c in ax.containers: ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.ylim(0, 1.05); plt.show()
print("Chênh lệch mAP@0.5:0.95 (A − B) = rò rỉ + khác biệt độ khó của val B (xem kết luận phần 5):", round(score.loc["A_original", "mAP@0.5:0.95"] - score.loc["B_video_split", "mAP@0.5:0.95"], 4))

In [ ]:
FINAL = "B_video_split"          # mô hình dùng cho phần còn lại
FINAL_MODEL, FINAL_DIR = MODELS[FINAL], RUN_DIRS[FINAL]

def show_imgs(paths, cols=2, size=8):
    paths = [p for p in paths if Path(p).exists()]
    if not paths: return
    rows = int(np.ceil(len(paths) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * .8 * rows))
    for ax, p in zip(np.ravel(axes), paths):
        ax.imshow(Image.open(p)); ax.set_title(Path(p).name); ax.axis("off")
    for ax in np.ravel(axes)[len(paths):]: ax.axis("off")
    plt.tight_layout(); plt.show()

def first_existing(d, names):
    return next((d / n for n in names if (d / n).exists()), d / names[0])

show_imgs([first_existing(FINAL_DIR, ["confusion_matrix_normalized.png", "confusion_matrix.png"]),
           first_existing(FINAL_DIR, ["BoxPR_curve.png", "PR_curve.png"]),
           first_existing(FINAL_DIR, ["BoxF1_curve.png", "F1_curve.png"]),
           first_existing(FINAL_DIR, ["BoxP_curve.png", "P_curve.png"])])
show_imgs(sorted(FINAL_DIR.glob("train_batch*.jpg"))[:2] + [FINAL_DIR / "val_batch0_labels.jpg", FINAL_DIR / "val_batch0_pred.jpg"])

### 7.3 Chọn ngưỡng confidence tối ưu (theo F1)

In [ ]:
m = METRICS[FINAL]
try:
    px, f1 = np.asarray(m.box.px), np.asarray(m.box.f1_curve).mean(0)
    p_c, r_c = np.asarray(m.box.p_curve).mean(0), np.asarray(m.box.r_curve).mean(0)
    BEST_CONF = float(px[f1.argmax()])
    plt.figure(figsize=(9, 4))
    plt.plot(px, f1, label="F1"); plt.plot(px, p_c, label="Precision"); plt.plot(px, r_c, label="Recall")
    plt.axvline(BEST_CONF, ls="--", c="k", label=f"best conf = {BEST_CONF:.3f}")
    plt.xlabel("confidence threshold"); plt.legend(); plt.title(f"Chọn ngưỡng — F1 max = {f1.max():.3f}"); plt.show()
except Exception as e:
    BEST_CONF = 0.25
    print("Không đọc được F1 curve:", e)
BEST_CONF = float(np.clip(BEST_CONF, 0.05, 0.9))
print("BEST_CONF =", round(BEST_CONF, 3))

### 7.4 Phân tích lỗi trên validation (B)
Ghép dự đoán với ground truth (IoU ≥ 0.5) để tính **TP / FP / FN**, recall theo kích thước box, và hiển thị những ảnh sai nhiều nhất.
Màu: <span style="color:green">GT bị bỏ sót (FN)</span>, <span style="color:blue">dự đoán đúng (TP)</span>, <span style="color:red">dự đoán sai (FP)</span>.

In [ ]:
def xywhn_to_xyxy(lab, W, H):
    cx, cy, w, h = lab[:, 1] * W, lab[:, 2] * H, lab[:, 3] * W, lab[:, 4] * H
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

val_root = Path(YAML_B).parent / "val"
val_imgs = sorted((val_root / "images").glob("*"))
err_rows, gt_rows, per_image = [], [], []
for r in FINAL_MODEL.predict([str(p) for p in val_imgs], imgsz=IMGSZ, conf=BEST_CONF, verbose=False, stream=True):
    p = Path(r.path); H_, W_ = r.orig_shape
    gl = np.loadtxt(val_root / "labels" / (p.stem + ".txt"), ndmin=2).reshape(-1, 5)
    gt = xywhn_to_xyxy(gl, W_, H_)
    pb, pc = r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()
    iou = iou_matrix(pb, gt); matched_gt, tp = set(), np.zeros(len(pb), bool)
    for i in np.argsort(-pc):
        if iou.shape[1] == 0: break
        j = int(np.argmax(np.where([k in matched_gt for k in range(len(gt))], -1, iou[i])))
        if iou[i, j] >= 0.5 and j not in matched_gt:
            tp[i] = True; matched_gt.add(j); err_rows.append(dict(type="TP", conf=pc[i], iou=iou[i, j]))
    for i in np.where(~tp)[0]: err_rows.append(dict(type="FP", conf=pc[i], iou=iou[i].max() if iou.shape[1] else 0))
    for j in range(len(gt)):
        gt_rows.append(dict(h_px=gt[j, 3] - gt[j, 1], w_px=gt[j, 2] - gt[j, 0], detected=j in matched_gt))
    per_image.append(dict(path=p, gt=gt, pb=pb, tp=tp, matched=matched_gt, fp=int((~tp).sum()), fn=len(gt) - len(matched_gt)))

err = pd.DataFrame(err_rows, columns=["type", "conf", "iou"])
gtdf, pim = pd.DataFrame(gt_rows), pd.DataFrame(per_image)
n_tp, n_fp, n_fn = (err.type == "TP").sum(), (err.type == "FP").sum(), (~gtdf.detected).sum()
print(f"conf={BEST_CONF:.3f} | TP={n_tp} FP={n_fp} FN={n_fn} | P={n_tp / max(n_tp + n_fp, 1):.3f} R={n_tp / max(n_tp + n_fn, 1):.3f}")

fig, ax = plt.subplots(1, 4, figsize=(22, 4))
ax[0].bar(["TP", "FP", "FN"], [n_tp, n_fp, n_fn], color=["C0", "C3", "C2"]); ax[0].set_title("TP / FP / FN")
if n_tp: ax[1].hist(err[err.type == "TP"].iou, bins=20, color="C0"); ax[1].set_title("IoU của các TP (độ chính xác định vị)")
for t, c in [("TP", "C0"), ("FP", "C3")]:
    if (err.type == t).any(): ax[2].hist(err[err.type == t].conf, bins=20, alpha=.6, label=t, color=c)
ax[2].set_title("Phân bố confidence TP vs FP"); ax[2].legend()
gtdf["h_bin"] = pd.cut(gtdf.h_px, bins=np.quantile(gtdf.h_px, [0, .2, .4, .6, .8, 1]), include_lowest=True)
rec = gtdf.groupby("h_bin").detected.mean()
ax[3].bar(range(len(rec)), rec.values, color="C4"); ax[3].set_xticks(range(len(rec)))
ax[3].set_xticklabels([f"{i.left:.0f}-{i.right:.0f}" for i in rec.index], rotation=20)
ax[3].set_ylim(0, 1.05); ax[3].set_title("Recall theo chiều cao box GT (px)")
plt.tight_layout(); plt.show()

In [ ]:
worst = pim.assign(err=pim.fp + pim.fn).sort_values("err", ascending=False)
worst = worst[worst.err > 0].head(8)
if len(worst) == 0:
    print("Không có ảnh nào sai ở ngưỡng này 🎉")
else:
    cols = 4; rows = int(np.ceil(len(worst) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    for ax, r in zip(np.ravel(axes), worst.to_dict("records")):
        ax.imshow(Image.open(r["path"])); ax.axis("off"); ax.set_title(f"FP={r['fp']} FN={r['fn']}\n{r['path'].name[:28]}", fontsize=8)
        for j, b in enumerate(r["gt"]):
            if j not in r["matched"]: ax.add_patch(patches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec="lime", lw=2.5))
        for b, t in zip(r["pb"], r["tp"]):
            ax.add_patch(patches.Rectangle(b[:2], b[2] - b[0], b[3] - b[1], fill=False, ec="blue" if t else "red", lw=1.5))
    for ax in np.ravel(axes)[len(worst):]: ax.axis("off")
    plt.suptitle("Ảnh sai nhiều nhất — xanh lá: FN, xanh dương: TP, đỏ: FP"); plt.tight_layout(); plt.show()

## 8. Inference trên tập test (không có nhãn)
Lưu ý: một số ảnh test trùng y hệt hoặc gần trùng với ảnh train (xem phần 4.3), nên kết quả trên những ảnh này sẽ đẹp hơn thực tế.

In [ ]:
test_meta = meta[meta.split == "test"].reset_index(drop=True)
test_meta["dup_in_train"] = test_meta.dup_group.isin(dedup.loc[dedup.subset == "train", "dup_group"])
preds = list(FINAL_MODEL.predict([str(p) for p in test_meta.path], imgsz=IMGSZ, conf=BEST_CONF,
                                 save=True, save_txt=True, save_conf=True, project=str(RUNS),
                                 name="test_pred", exist_ok=True, verbose=False, stream=True))
rows = []
for p, r in zip(test_meta.path, preds):
    for b, c, cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        rows.append(dict(image=p.name, cls=CLASS_NAMES[int(c)], conf=float(cf), x1=b[0], y1=b[1], x2=b[2], y2=b[3]))
pred_df = pd.DataFrame(rows, columns=["image", "cls", "conf", "x1", "y1", "x2", "y2"])
pred_df.to_csv(WORK / "test_predictions.csv", index=False)
test_meta["n_pred"] = test_meta.name.map(pred_df.image.value_counts()).fillna(0).astype(int)
print(f"{len(test_meta)} ảnh test | {len(pred_df)} box | ảnh không có box: {(test_meta.n_pred == 0).sum()} | "
      f"ảnh test trùng y hệt ảnh train: {test_meta.dup_in_train.sum()}")

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ax[0].hist(pred_df.conf, bins=20, range=(0, 1), color="C0"); ax[0].axvline(BEST_CONF, ls="--", c="k"); ax[0].set_title("Confidence của dự đoán trên test")
ax[1].hist(test_meta.n_pred, bins=np.arange(-.5, test_meta.n_pred.max() + 1.5), color="C1"); ax[1].set_title("Số box dự đoán / ảnh test")
ax[2].hist(boxes[boxes.split == "train"].groupby("img").size(), bins=np.arange(-.5, 8.5), alpha=.5, density=True, label="train GT")
ax[2].hist(test_meta.n_pred, bins=np.arange(-.5, 8.5), alpha=.5, density=True, label="test pred"); ax[2].legend()
ax[2].set_title("So sánh số box/ảnh: GT train vs dự đoán test")
plt.tight_layout(); plt.show()

sample = random.sample(preds, min(8, len(preds)))
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax, r in zip(np.ravel(axes), sample):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off"); ax.set_title(Path(r.path).name[:28], fontsize=8)
for ax in np.ravel(axes)[len(sample):]: ax.axis("off")
plt.suptitle(f"Dự đoán trên test (conf ≥ {BEST_CONF:.2f})"); plt.tight_layout(); plt.show()

## 9. Lưu kết quả

In [ ]:
best_pt = FINAL_DIR / "weights" / "best.pt"
shutil.copy(best_pt, WORK / "motorbike_yolov8_best.pt")
try:
    onnx_path = YOLO(str(best_pt)).export(format="onnx", imgsz=IMGSZ, simplify=True)
    shutil.copy(onnx_path, WORK / "motorbike_yolov8_best.onnx")
except Exception as e:
    print("ONNX export lỗi:", e)
score.assign(best_conf=BEST_CONF).to_csv(WORK / "val_metrics.csv")
for k, d in RUN_DIRS.items():
    shutil.copy(d / "results.csv", WORK / f"results_{k}.csv")
shutil.rmtree(DS_ROOT, ignore_errors=True)    # xoá bản sao dataset cho output gọn
for f in sorted(WORK.glob("*")):
    print(f.name)

## Tóm tắt
- **Preprocessing:** làm sạch nhãn, loại frame trùng, chia train/val theo video để loại bỏ rò rỉ; augmentation chọn theo đặc điểm dữ liệu (box cao và hẹp, ánh sáng thay đổi, ít bối cảnh).
- **Đánh giá:** mAP của **B** là ước lượng trung thực hơn cho video mới. Chênh lệch A − B cho thấy split gốc lạc quan đến mức nào, nhưng cần trừ hao vì val B đông xe hơn và chỉ có 2 video. mAP@0.5:0.95 bị giới hạn bởi nhiễu nhãn (IoU giữa 2 lần gán nhãn ≈ 0,89).
- **Đầu ra** (`/kaggle/working`): `motorbike_yolov8_best.pt`, `.onnx`, `val_metrics.csv`, `results_*.csv`, `test_predictions.csv`, `runs/`.